# UNAM vs UAG — RAG Arena: Competencia (Google Colab T4)

**Reglas de la competencia:**
- **Modelo obligatorio (`gemma3:4b`):** todos los equipos compiten con `gemma3:4b` local en Ollama. Prohibido llamar APIs comerciales (`OpenAI`, `Gemini`, `Anthropic`, etc.).
- **Corpus y preguntas en inglés:** tu `build_prompt` debe responder en inglés de forma concisa (máx. 600 chars).
- **Ejecución en 3 clics (sin editar código en Colab):**
    1. **Celda 1** (1 vez por sesión): prepara Ollama (`gemma3:4b`) y extrae `RAW_DOCUMENTS`.
    2. **Celda 2**: sube tu archivo `.py` (`class MyRAG`), indexa el corpus y valida `LOCAL_EVAL_SET`.
    3. **Celda 3**: pega únicamente tu `TEAM_TOKEN` y conéctate (detecta automáticamente modo práctica o show en vivo).

In [ ]:
# @title Celda 1: Preparar entorno, corpus y gemma3:4b
import json
from pathlib import Path
import shutil
import subprocess
import sys
import time
from typing import Any
from urllib.error import URLError
from urllib.request import urlopen, urlretrieve
import zipfile

OLLAMA_MODEL = "gemma3:4b"
OLLAMA_NUM_CTX = 4096
CORPUS_ZIP_PATH = "corpus_hackathon.zip"
CORPUS_ZIP_URL = "https://storage.googleapis.com/kajuit-hackathon-2026-assets/corpus_hackathon.zip"
ARENA_SDK_URL = "https://storage.googleapis.com/kajuit-hackathon-2026-assets/kajuit_sdk.py"
OLLAMA_URL = "http://127.0.0.1:11434"

# 1. Instalar dependencias base del runtime
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "websockets>=13.0,<16.0",
    "sentence-transformers",
    "numpy",
])

# 2. Instalar e iniciar Ollama en Colab T4 (idempotente)
if shutil.which("ollama") is None:
    if shutil.which("zstd") is None:
        subprocess.run("apt-get update -qq && apt-get install -y -qq zstd", shell=True, check=True)
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

def _ollama_models() -> list[str]:
    try:
        with urlopen(f"{OLLAMA_URL}/api/tags", timeout=2.0) as resp:
            if resp.status != 200:
                return []
            data = json.loads(resp.read().decode("utf-8"))
            return [str(m.get("name", "")) for m in data.get("models", [])]
    except (URLError, TimeoutError, OSError, json.JSONDecodeError):
        return []

def _ollama_ready() -> bool:
    try:
        with urlopen(f"{OLLAMA_URL}/api/tags", timeout=2.0) as resp:
            return resp.status == 200
    except (URLError, TimeoutError, OSError):
        return False

if not _ollama_ready():
    with open("ollama_serve.log", "ab") as ollama_log:
        subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=ollama_log)
    for _ in range(30):
        if _ollama_ready():
            break
        time.sleep(1.0)
    if not _ollama_ready():
        raise RuntimeError("No se pudo iniciar Ollama. Revisa ollama_serve.log.")

local_models = _ollama_models()
if OLLAMA_MODEL not in local_models and f"{OLLAMA_MODEL}:latest" not in local_models:
    subprocess.run(["ollama", "pull", OLLAMA_MODEL], check=True)

# 3. Descargar SDK de la arena y corpus_hackathon.zip
def _ensure_file(local_path: str, download_url: str, label: str) -> Path:
    target = Path(local_path)
    if target.exists():
        return target
    if download_url.strip():
        print(f"Descargando {label}...")
        urlretrieve(download_url.strip(), target)
        return target
    raise FileNotFoundError(
        f"Falta {target}. Proporciona una URL o coloca {target.name} en el runtime."
    )

if not Path("kajuit_sdk.py").exists():
    if Path("student/kajuit_sdk.py").exists():
        sys.path.insert(0, str(Path("student").resolve()))
    else:
        _ensure_file("kajuit_sdk.py", ARENA_SDK_URL, "SDK de la arena")
corpus_zip = _ensure_file(CORPUS_ZIP_PATH, CORPUS_ZIP_URL, "corpus del torneo")

# 4. Extraer documentos .txt crudos
RAW_DOCUMENTS: list[str] = []
with zipfile.ZipFile(corpus_zip, "r") as zf:
    for member in sorted(zf.namelist()):
        if member.endswith(".txt"):
            text = zf.read(member).decode("utf-8", errors="replace").strip()
            if text:
                RAW_DOCUMENTS.append(text)

if not RAW_DOCUMENTS:
    raise ValueError(f"No se encontraron archivos .txt en {corpus_zip}")

# 5. Utilidad local de evaluación con diagnóstico detallado por pregunta
def evaluate_local_rag(
    eval_items: list[dict[str, str]],
    rag_or_fn: Any,
    *,
    k: int | None = None,
    max_detail_items: int = 25,
) -> dict[str, float]:
    """Mide Recall, Accuracy, latencia y muestra diagnóstico por pregunta."""
    if not eval_items:
        print(
            "LOCAL_EVAL_SET está vacío en tu archivo .py; "
            "agrega pares {'question': ..., 'expected': ...} para evaluar localmente."
        )
        return {"recall_at_k": 0.0, "accuracy": 0.0, "avg_ms": 0.0, "max_ms": 0.0}

    predict_fn = rag_or_fn.predict if hasattr(rag_or_fn, "predict") else rag_or_fn
    retrieve_fn = getattr(rag_or_fn, "retrieve", None)
    recall_hits = 0
    answer_hits = 0
    latencies_ms: list[float] = []
    records: list[dict[str, Any]] = []

    for idx, item in enumerate(eval_items, start=1):
        question = str(item["question"])
        expected_raw = str(item["expected"]).strip()
        expected_low = expected_raw.lower()
        passages: list[str] = []
        recalled = False
        if callable(retrieve_fn):
            raw_passages = retrieve_fn(question, k=k) if k is not None else retrieve_fn(question)
            passages = [str(p) for p in raw_passages]
            recalled = bool(expected_low) and any(expected_low in p.lower() for p in passages)
            if recalled:
                recall_hits += 1

        t0 = time.perf_counter()
        pred_raw = str(predict_fn(question)).strip()
        elapsed_ms = (time.perf_counter() - t0) * 1000.0
        latencies_ms.append(elapsed_ms)
        matched = bool(expected_low) and expected_low in pred_raw.lower()
        if matched:
            answer_hits += 1

        records.append({
            "idx": idx,
            "question": question,
            "expected": expected_raw,
            "pred": pred_raw,
            "n_passages": len(passages),
            "recalled": recalled,
            "matched": matched,
            "ms": elapsed_ms,
        })

    total = len(eval_items)
    if total <= max_detail_items:
        to_show = records
        print(f"Detalle por pregunta ({total}):")
    else:
        failures = [r for r in records if not (r["recalled"] and r["matched"])]
        to_show = failures[:max_detail_items]
        print(
            f"Detalle de errores ({len(failures)} con fallo de {total}; "
            f"mostrando hasta {max_detail_items}):"
        )

    for rec in to_show:
        recall_lbl = (
            f"HIT ({rec['n_passages']} pasajes)"
            if rec["recalled"]
            else f"MISS ({rec['n_passages']} pasajes)"
        )
        match_lbl = "OK" if rec["matched"] else "FAIL"
        pred_preview = rec["pred"].replace("\n", " ")[:180] or "<vacío>"
        print(
            f"  [{rec['idx']}] {rec['question']}\n"
            f"      Recall: {recall_lbl} | Match: {match_lbl} | {rec['ms']:.1f} ms\n"
            f"      Esperado: {rec['expected']}\n"
            f"      Modelo:   {pred_preview}"
        )

    summary = {
        "recall_at_k": recall_hits / total,
        "accuracy": answer_hits / total,
        "avg_ms": sum(latencies_ms) / total,
        "max_ms": max(latencies_ms),
    }
    print(
        f"Resumen local (n={total}): "
        f"Recall={summary['recall_at_k']:.0%}, "
        f"Accuracy={summary['accuracy']:.0%}, "
        f"Promedio={summary['avg_ms']:.1f} ms, Máxima={summary['max_ms']:.1f} ms"
    )
    return summary

# 6. Precargar el modelo en la VRAM de la T4 para evitar TimeoutError en la Ronda 1
from kajuit_sdk import ollama_generate

t_warm = time.perf_counter()
_warm_reply = ollama_generate(
    "Reply with OK.",
    model=OLLAMA_MODEL,
    ollama_url=OLLAMA_URL,
    num_predict=4,
    num_ctx=OLLAMA_NUM_CTX,
    timeout_seconds=240.0,
)
if not _warm_reply:
    raise RuntimeError("Falló el calentamiento de Ollama o expiró el tiempo. Revisa ollama_serve.log.")

print(
    f"Listo: {len(RAW_DOCUMENTS)} documentos extraídos; "
    f"Ollama '{OLLAMA_MODEL}' cargado en VRAM en {time.perf_counter() - t_warm:.1f}s."
)

In [ ]:
# @title Celda 2: Subir tu archivo .py (MyRAG) y evaluar

import ast
import hashlib
from importlib import metadata as importlib_metadata
import logging
from pathlib import Path
import re
import shlex
import subprocess
import sys
import time
import types
from typing import Any

from kajuit_sdk import BaseRAGPipeline, ollama_generate

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)

if "RAW_DOCUMENTS" not in globals():
    raise RuntimeError("Ejecuta primero la Celda 1 para descargar el corpus y calentar Ollama.")

_VERSION_SPEC_RE = re.compile(r"(==|>=|<=|!=|~=|<|>|@|://)")

def _extract_extra_pip_packages(source_text: str) -> list[str]:
    """Lee EXTRA_PIP_PACKAGES del módulo .py por AST sin ejecutarlo aún."""
    try:
        tree = ast.parse(source_text)
    except SyntaxError:
        return []
    for node in tree.body:
        target_name: str | None = None
        value_node: ast.AST | None = None
        if isinstance(node, ast.Assign) and len(node.targets) == 1 and isinstance(node.targets[0], ast.Name):
            target_name = node.targets[0].id
            value_node = node.value
        elif isinstance(node, ast.AnnAssign) and isinstance(node.target, ast.Name):
            target_name = node.target.id
            value_node = node.value
        if target_name == "EXTRA_PIP_PACKAGES" and value_node is not None:
            try:
                raw_val = ast.literal_eval(value_node)
            except (ValueError, SyntaxError):
                return []
            if isinstance(raw_val, str):
                return shlex.split(raw_val.strip())
            if isinstance(raw_val, (list, tuple)):
                return [str(item).strip() for item in raw_val if isinstance(item, str) and item.strip()]
            return []
    return []

def _filter_missing_pip_packages(specs: list[str]) -> tuple[list[str], list[str]]:
    """Omite paquetes sin pin de versión ya presentes en el entorno."""
    to_install: list[str] = []
    already_present: list[str] = []
    for raw_spec in specs:
        spec = raw_spec.strip()
        if not spec:
            continue
        if _VERSION_SPEC_RE.search(spec):
            to_install.append(spec)
            continue
        pkg_name = spec.split("[", 1)[0].strip()
        try:
            importlib_metadata.distribution(pkg_name)
            already_present.append(spec)
        except importlib_metadata.PackageNotFoundError:
            to_install.append(spec)
    return to_install, already_present

def _install_missing_packages(specs: list[str]) -> None:
    to_install, already_present = _filter_missing_pip_packages(specs)
    if already_present:
        print(f"Paquetes ya disponibles (omitidos): {', '.join(already_present)}")
    if to_install:
        print(f"Instalando paquetes adicionales: {', '.join(to_install)}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *to_install])

def _import_my_rag_from_path(module_path: Path) -> tuple[Any, type[Any]]:
    """Instala EXTRA_PIP_PACKAGES faltantes, compila desde fuente sin caché .pyc y valida MyRAG."""
    source_text = module_path.read_text(encoding="utf-8")
    _install_missing_packages(_extract_extra_pip_packages(source_text))
    digest = hashlib.sha256(source_text.encode("utf-8")).hexdigest()[:8]
    mod_name = f"_arena_team_rag_{digest}"
    module = types.ModuleType(mod_name)
    module.__file__ = str(module_path.resolve())
    sys.modules[mod_name] = module
    code_obj = compile(source_text, str(module_path), "exec")
    exec(code_obj, module.__dict__)
    rag_cls = getattr(module, "MyRAG", None)
    if not isinstance(rag_cls, type):
        raise TypeError(f"El archivo '{module_path.name}' debe definir `class MyRAG:`.")
    if not issubclass(rag_cls, BaseRAGPipeline):
        missing = [m for m in ("fit", "retrieve", "build_prompt") if not callable(getattr(rag_cls, m, None))]
        if missing:
            raise TypeError(
                f"`MyRAG` en '{module_path.name}' debe implementar: {', '.join(missing)}."
            )
    return module, rag_cls

def _build_rag_adapter(
    rag_cls: type[Any],
    *,
    model: str,
    ollama_url: str,
    num_ctx: int,
) -> Any:
    """Envuelve MyRAG plana o BaseRAGPipeline con una interfaz uniforme .fit()/.retrieve()/.predict()."""
    if issubclass(rag_cls, BaseRAGPipeline):
        return rag_cls(model=model, ollama_url=ollama_url, num_ctx=num_ctx)

    inner = rag_cls()

    class _StandaloneRAGAdapter:
        def __init__(self, target: Any) -> None:
            self._target = target

        def fit(self, raw_documents: list[str]) -> "_StandaloneRAGAdapter":
            fitted = self._target.fit(raw_documents)
            if fitted is None:
                raise TypeError(
                    "`MyRAG.fit(raw_documents)` devolvió None; asegúrate de terminar con `return self`."
                )
            self._target = fitted
            return self

        def retrieve(self, query: str) -> list[str]:
            passages = self._target.retrieve(query)
            if not isinstance(passages, list):
                raise TypeError("`MyRAG.retrieve(query)` debe devolver `list[str]`.")
            return [str(p) for p in passages]

        def build_prompt(self, question: str, passages: list[str]) -> str:
            prompt = self._target.build_prompt(question, passages)
            if not isinstance(prompt, str):
                raise TypeError("`MyRAG.build_prompt(question, passages)` debe devolver `str`.")
            return prompt

        def predict(self, question: str) -> str:
            if callable(getattr(self._target, "predict", None)):
                return str(self._target.predict(question))
            passages = self.retrieve(question)
            prompt = self.build_prompt(question, passages)
            return ollama_generate(
                prompt,
                model=model,
                ollama_url=ollama_url,
                num_ctx=num_ctx,
            )

    return _StandaloneRAGAdapter(inner)


UPLOADED_RAG_PATH = Path("_team_rag_uploaded.py")

def _save_uploaded_python_bytes(uploaded: dict[str, bytes], target: Path) -> tuple[str, str]:
    """Guarda los bytes del primer archivo .py subido sobrescribiendo siempre `target`."""
    py_items = [(name, raw) for name, raw in uploaded.items() if name.lower().endswith(".py")]
    if not py_items:
        raise ValueError("El archivo subido debe tener extensión .py (ej. my_rag.py).")
    orig_name, raw_bytes = py_items[0]
    target.write_bytes(raw_bytes)
    orig_path = Path(orig_name)
    if orig_path.exists() and orig_path.resolve() != target.resolve():
        orig_path.unlink(missing_ok=True)
    digest = hashlib.sha256(raw_bytes).hexdigest()[:8]
    return orig_name, digest

try:
    from google.colab import files  # type: ignore[import-not-found]
    print("Sube tu archivo .py con `class MyRAG:` (cualquier nombre .py):")
    uploaded_files = files.upload()
    if uploaded_files:
        orig_name, digest = _save_uploaded_python_bytes(uploaded_files, UPLOADED_RAG_PATH)
        print(f"Archivo cargado: {orig_name} (sha256: {digest})")
        active_rag_path = UPLOADED_RAG_PATH
    elif UPLOADED_RAG_PATH.exists():
        digest = hashlib.sha256(UPLOADED_RAG_PATH.read_bytes()).hexdigest()[:8]
        print(f"Subida cancelada; reutilizando el archivo cargado previamente (sha256: {digest}).")
        active_rag_path = UPLOADED_RAG_PATH
    else:
        raise FileNotFoundError("Debes subir un archivo .py que defina `class MyRAG:`.")
except ImportError:
    candidates = [UPLOADED_RAG_PATH, Path("my_rag.py"), Path("student/my_rag.py")]
    active_rag_path = next((p for p in candidates if p.exists()), UPLOADED_RAG_PATH)
    if not active_rag_path.exists():
        raise FileNotFoundError("No se encontró un archivo .py local (`my_rag.py` o `student/my_rag.py`).")
    digest = hashlib.sha256(active_rag_path.read_bytes()).hexdigest()[:8]
    print(f"Archivo local cargado: {active_rag_path.name} (sha256: {digest})")

team_module, MyRAG = _import_my_rag_from_path(active_rag_path)

t_fit = time.perf_counter()
rag = _build_rag_adapter(
    MyRAG,
    model=OLLAMA_MODEL,
    ollama_url=OLLAMA_URL,
    num_ctx=OLLAMA_NUM_CTX,
).fit(RAW_DOCUMENTS)
if rag is None:
    raise TypeError("`MyRAG.fit(raw_documents)` devolvió None; asegúrate de terminar con `return self`.")
print(f"MyRAG.fit() completado en {time.perf_counter() - t_fit:.1f}s.")

def answer_question(question: str) -> str:
    return rag.predict(question)

LOCAL_EVAL_SET: list[dict[str, str]] = list(getattr(team_module, "LOCAL_EVAL_SET", []))
evaluate_local_rag(LOCAL_EVAL_SET, rag)

In [ ]:
# @title Celda 3: Conectar a la RAG Arena (re-ejecutable sin re-indexar)
import asyncio
from datetime import datetime, timedelta, timezone
import json
import logging
from urllib.error import URLError
from urllib.request import urlopen

from kajuit_sdk import KajuitAuthError, KajuitClient

TEAM_TOKEN = "REPLACE_WITH_TEAM_TOKEN"  # @param {type:"string"}

_ARENA_HTTP_URL = "https://kajuit-arena-1061984402706.us-central1.run.app"
_ARENA_WS_URL = "wss://kajuit-arena-1061984402706.us-central1.run.app"
_LIVE_START_AT = datetime(2026, 11, 28, 8, 0, tzinfo=timezone(timedelta(hours=-6)))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)

if "answer_question" not in globals():
    raise RuntimeError("Ejecuta primero la Celda 1 y la Celda 2 para inicializar `answer_question` y el índice.")

def _resolve_max_rounds(
    http_url: str = _ARENA_HTTP_URL,
    *,
    live_start_at: datetime = _LIVE_START_AT,
    now: datetime | None = None,
) -> tuple[int, str]:
    """Detecta modo practice vs live desde /api/stage con fallback por fecha."""
    try:
        with urlopen(f"{http_url.rstrip('/')}/api/stage", timeout=4.0) as resp:
            if resp.status == 200:
                stage = json.loads(resp.read().decode("utf-8"))
                mode = str(stage.get("game_mode", "")).strip().lower()
                if mode == "live":
                    return 0, "live (servidor)"
                if mode == "practice":
                    total = max(1, int(stage.get("total_rounds") or 10))
                    return total, f"practice ({total} rondas, servidor)"
    except (URLError, TimeoutError, OSError, ValueError, json.JSONDecodeError):
        pass

    current = now if now is not None else datetime.now(timezone.utc)
    if current.tzinfo is None:
        current = current.replace(tzinfo=timezone.utc)
    if current >= live_start_at.astimezone(timezone.utc):
        return 0, "live (fecha del evento)"
    return 10, "practice (10 rondas, fecha previa)"

cleaned_token = TEAM_TOKEN.strip()
if not cleaned_token or "REPLACE_WITH_TEAM_TOKEN" in cleaned_token:
    print("Configura tu TEAM_TOKEN antes de conectarte a la RAG Arena.")
else:
    max_rounds, mode_label = _resolve_max_rounds()
    print(f"Conectando a la RAG Arena en modo {mode_label}...")
    _practice_rounds_done = 0
    _client_ref: list[KajuitClient] = []

    def _on_arena_event(event_name: str, detail: dict[str, object]) -> None:
        global _practice_rounds_done
        if max_rounds > 0 and event_name == "ack_received":
            if detail.get("status") in ("accepted", "quota_exceeded"):
                _practice_rounds_done += 1
                if _practice_rounds_done >= max_rounds and _client_ref:
                    print(
                        f"Completadas {max_rounds} rondas de práctica. "
                        "Ajusta tu .py si lo deseas, re-ejecuta la Celda 2 y vuelve a correr la Celda 3."
                    )
                    _client_ref[0].request_stop()

    client = KajuitClient(
        server_url=_ARENA_WS_URL,
        team_token=cleaned_token,
        answer_fn=answer_question,
        on_event=_on_arena_event,
    )
    _client_ref.append(client)
    try:
        await client.run()
    except (KeyboardInterrupt, asyncio.CancelledError):
        await client.close()
        print("Desconectado por el usuario.")
    except KajuitAuthError as exc:
        print(f"Error de autenticación / sesión duplicada: {exc}")